# Clase 20 — Interpretacion de Clusters, K-Means 3D y Pipelines

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador × UDLA

---

**Objetivos de hoy:**
1. **Interpretar** los 5 segmentos de clientes encontrados con K-Means (2D).
2. Agregar **Edad** como tercera variable y visualizar en **3D con Plotly**.
3. Aplicar K-Means a un dataset **sin etiquetas** (ejercicio Iris).
4. Construir **Pipelines de sklearn** para encadenar preprocesamiento + modelo.

**Datasets:**
- Mall Customers (secciones 1-3)
- Iris sin etiquetas (seccion 4)
- Pima Indians Diabetes (seccion 5 — Pipelines)

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

print("Imports OK")

---
## 1. Retomar: Segmentacion de clientes con K-Means (2D)

En la clase pasada encontramos 5 segmentos de clientes usando **income** y **spending**. Hoy vamos a **interpretar** esos segmentos: darles nombres de negocio, entender que los diferencia y generar recomendaciones accionables.

### 1.1 Cargar datos y aplicar K-Means con K=5

In [ ]:
URL_MALL = "https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-19/mall_customers.csv"
df = pd.read_csv(URL_MALL)
df = df.rename(columns={
    "Annual Income (k$)": "income",
    "Spending Score (1-100)": "spending"
})

# K-Means con K=5 (lo que encontramos con el metodo del codo)
X_2d = df[["income", "spending"]].values
km_2d = KMeans(n_clusters=5, random_state=42, n_init=10)
df["cluster"] = km_2d.fit_predict(X_2d)

print(f"Shape: {df.shape}")
print(f"Clientes por cluster:\n{df['cluster'].value_counts().sort_index()}")

In [ ]:
# Scatter 2D con clusters
COLORS = ["#2563EB", "#C82B40", "#16A34A", "#EA580C", "#7C3AED"]

fig = px.scatter(
    df, x="income", y="spending",
    color=df["cluster"].astype(str),
    title="5 Segmentos de Clientes — K-Means (2D)",
    labels={"income": "Ingreso anual (k$)", "spending": "Spending Score", "color": "Cluster"},
    color_discrete_sequence=COLORS, opacity=0.8)

# Centroides
for i, c in enumerate(km_2d.cluster_centers_):
    fig.add_trace(go.Scatter(
        x=[c[0]], y=[c[1]], mode="markers",
        marker=dict(size=15, symbol="x", color=COLORS[i],
                    line=dict(width=2, color="black")),
        name=f"Centroide {i}", showlegend=False))

fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los 5 clusters estan bien separados en el espacio ingreso-gasto. Ahora la pregunta clave: **que significan para el negocio?** Los numeros de cluster (0, 1, 2...) no dicen nada. Necesitamos **perfilarlos**.

### 1.2 Profiling — Promedios por cluster

El primer paso para interpretar clusters es calcular las **estadisticas promedio** de cada grupo. Esto nos dice que tiene de especial cada segmento.

In [ ]:
# Perfil numerico por cluster
perfil = df.groupby("cluster")[["Age", "income", "spending"]].agg(["mean", "std", "count"])
perfil.columns = [f"{col}_{stat}" for col, stat in perfil.columns]
perfil = perfil.round(1)
perfil

**Interpretacion:** Cada cluster tiene un perfil distinto en ingreso y gasto. La desviacion estandar nos dice que tan homogeneo es cada grupo — si es baja, los miembros del cluster son parecidos entre si.

### 1.3 Nombres de negocio

Los numeros de cluster no significan nada. Vamos a darles **nombres que un gerente de marketing entienda**.

In [ ]:
# Asignar nombres basados en ingreso y gasto promedio
perfil_simple = df.groupby("cluster")[["income", "spending"]].mean()

nombres = {}
for c in perfil_simple.index:
    inc = perfil_simple.loc[c, "income"]
    sp  = perfil_simple.loc[c, "spending"]
    if inc > 70 and sp > 60:
        nombres[c] = "Premium"
    elif inc > 70 and sp < 40:
        nombres[c] = "Ahorradores VIP"
    elif inc < 40 and sp > 60:
        nombres[c] = "Entusiastas"
    elif inc < 40 and sp < 40:
        nombres[c] = "Cautelosos"
    else:
        nombres[c] = "Mainstream"

df["segmento"] = df["cluster"].map(nombres)
print("Segmentos encontrados:")
print(df["segmento"].value_counts())

In [ ]:
# Scatter 2D con nombres de negocio
fig = px.scatter(
    df, x="income", y="spending", color="segmento",
    title="Segmentos de Clientes con nombres de negocio",
    labels={"income": "Ingreso anual (k$)", "spending": "Spending Score"},
    color_discrete_map={
        "Premium": "#16A34A", "Ahorradores VIP": "#2563EB",
        "Entusiastas": "#EA580C", "Cautelosos": "#7C3AED",
        "Mainstream": "#6B7280"},
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Ahora cada punto tiene un nombre que un gerente entiende. No estamos hablando de "cluster 3" sino de "Ahorradores VIP" — personas con alto ingreso que no gastan. Esa es la **oportunidad de negocio mas grande**.

### 1.4 Visualizaciones para presentar al negocio

In [ ]:
# Barras agrupadas: ingreso y gasto promedio por segmento
perfil_seg = df.groupby("segmento")[["income", "spending", "Age"]].mean().round(1)
perfil_seg = perfil_seg.sort_values("income", ascending=False)

fig = px.bar(
    perfil_seg.reset_index(), x="segmento", y=["income", "spending"],
    barmode="group",
    title="Ingreso y Gasto promedio por segmento",
    labels={"value": "Promedio", "variable": "Variable"},
    color_discrete_map={"income": "#2563EB", "spending": "#C82B40"})
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** La brecha entre las barras azules (ingreso) y rojas (gasto) revela la oportunidad. Los "Ahorradores VIP" tienen la mayor brecha: ganan mucho pero gastan poco. Los "Entusiastas" tienen la brecha inversa: gastan mas de lo que su ingreso sugeriria.

In [ ]:
# Radar chart: perfil multivariable de cada segmento
# Normalizamos para que todas las variables queden en la misma escala (0-1)
perfil_radar = df.groupby("segmento")[["Age", "income", "spending"]].mean()
perfil_norm = (perfil_radar - perfil_radar.min()) / (perfil_radar.max() - perfil_radar.min())

categorias = list(perfil_norm.columns) + [perfil_norm.columns[0]]  # cerrar el poligono

colores = {
    "Premium": "#16A34A", "Ahorradores VIP": "#2563EB",
    "Entusiastas": "#EA580C", "Cautelosos": "#7C3AED", "Mainstream": "#6B7280"}

fig = go.Figure()
for seg in perfil_norm.index:
    valores = list(perfil_norm.loc[seg]) + [perfil_norm.loc[seg].iloc[0]]
    fig.add_trace(go.Scatterpolar(
        r=valores, theta=categorias, fill="toself",
        name=seg, line_color=colores[seg], opacity=0.6))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Radar: Perfil de cada segmento (normalizado 0-1)",
    template="plotly_white")
fig.show()

**Interpretacion:** El radar chart muestra de un vistazo la "forma" de cada segmento. Los Premium tienen ingreso y gasto altos. Los Cautelosos tienen todo bajo. Los Entusiastas tienen gasto alto pero ingreso bajo — son jovenes apasionados por la marca. Este tipo de grafica es muy util para presentaciones ejecutivas.

In [ ]:
# Distribucion de genero por segmento
gender_seg = df.groupby(["segmento", "Gender"]).size().reset_index(name="count")
fig = px.bar(
    gender_seg, x="segmento", y="count", color="Gender",
    barmode="group", title="Composicion de genero por segmento",
    color_discrete_map={"Male": "#2563EB", "Female": "#C82B40"})
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Si un segmento tiene composicion de genero muy diferente, las campanias de marketing pueden personalizarse. Por ejemplo, si los Entusiastas son mayoritariamente mujeres, las promociones para ese segmento pueden diseñarse con ese perfil en mente.

### 1.5 Distribucion de edad por segmento

In [ ]:
# Boxplot de edad por segmento
fig = px.box(
    df, x="segmento", y="Age", color="segmento",
    title="Distribucion de Edad por segmento",
    labels={"Age": "Edad", "segmento": "Segmento"},
    color_discrete_map={
        "Premium": "#16A34A", "Ahorradores VIP": "#2563EB",
        "Entusiastas": "#EA580C", "Cautelosos": "#7C3AED",
        "Mainstream": "#6B7280"})
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

**Interpretacion:** La edad agrega una dimension que K-Means no uso directamente (solo uso income y spending). Si vemos que algunos segmentos tienen edades muy distintas, significa que la edad **si importa** y deberiamos incluirla en el clustering. Eso es exactamente lo que haremos en la seccion 2.

### 1.6 Recomendaciones de negocio por segmento

La segmentacion no sirve si no genera **acciones concretas**. Aqui va una tabla de estrategias por segmento:

In [ ]:
# Tabla de recomendaciones
recomendaciones = pd.DataFrame({
    "Segmento": ["Premium", "Ahorradores VIP", "Entusiastas", "Cautelosos", "Mainstream"],
    "Perfil": [
        "Ingreso alto, gasto alto",
        "Ingreso alto, gasto bajo",
        "Ingreso bajo, gasto alto",
        "Ingreso bajo, gasto bajo",
        "Ingreso y gasto medio"
    ],
    "Estrategia": [
        "Programa de lealtad exclusivo, productos premium, experiencias VIP",
        "Incentivar gasto: descuentos, tarjeta, beneficios por compra",
        "Proteger: son leales pero sensibles al precio, no subir precios",
        "Bajo potencial inmediato: comunicacion basica, ofertas masivas",
        "Upselling gradual: moverlos hacia Premium o Entusiastas"
    ],
    "Prioridad": ["Retener", "Activar", "Proteger", "Monitorear", "Desarrollar"]
})

recomendaciones.style.set_properties(**{
    "text-align": "left", "white-space": "pre-wrap"
}).set_table_styles([
    {"selector": "th", "props": [("text-align", "left")]}
])

**Interpretacion:** La segmentacion transforma datos en **decisiones**. Cada segmento recibe una estrategia diferente. Sin K-Means, tratariamos a los 200 clientes igual — con K-Means, personalizamos las acciones. Eso es el valor real del clustering en negocios.

---
## 2. K-Means en 3 dimensiones — Agregar Edad

Hasta ahora usamos solo 2 variables (income, spending). Pero vimos en el boxplot que la **edad** varia entre segmentos. Si la incluimos, K-Means podra encontrar grupos mas finos.

**Problema:** income va de 15 a 137, spending de 1 a 99, pero Age va de 18 a 70. Las escalas son distintas. Si no las igualamos, K-Means le dara mas peso a la variable con valores mas grandes.

**Solucion:** `StandardScaler` — transforma cada variable para que tenga media 0 y desviacion estandar 1.

### 2.1 Escalar las 3 variables

In [ ]:
# Escalar las 3 variables
scaler = StandardScaler()
X_3d_raw = df[["income", "spending", "Age"]]
X_3d = scaler.fit_transform(X_3d_raw)

# Verificar que la escala quedo uniforme
print("Antes de escalar:")
print(X_3d_raw.describe().loc[["mean", "std"]].round(1))
print("\nDespues de escalar:")
print(pd.DataFrame(X_3d, columns=["income", "spending", "Age"]).describe().loc[["mean", "std"]].round(2))

**Interpretacion:** Despues de escalar, todas las variables tienen media ~0 y desviacion ~1. Ahora K-Means tratara las 3 variables con **igual importancia** al calcular distancias.

### 2.2 Encontrar K optimo (Codo + Silhouette)

In [ ]:
# Metodo del codo + Silhouette para 3D
inertias_3d = []
sil_3d = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_3d)
    inertias_3d.append(km.inertia_)
    sil_3d.append(silhouette_score(X_3d, labels))

# Graficar ambos en subplots
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Metodo del Codo (3D)", "Silhouette Score (3D)"))

fig.add_trace(go.Scatter(
    x=list(K_range), y=inertias_3d, mode="lines+markers",
    marker=dict(color="#2563EB")), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(K_range), y=sil_3d, mode="lines+markers",
    marker=dict(color="#C82B40")), row=1, col=2)

fig.update_xaxes(title_text="K", row=1, col=1)
fig.update_xaxes(title_text="K", row=1, col=2)
fig.update_yaxes(title_text="Inercia", row=1, col=1)
fig.update_yaxes(title_text="Silhouette", row=1, col=2)
fig.update_layout(template="plotly_white", showlegend=False,
                  title_text="Seleccion de K para clustering 3D")
fig.show()

**Interpretacion:** Al agregar edad, el K optimo puede cambiar. El codo y el Silhouette pueden sugerir un K diferente a 5. Esto es normal: mas variables = mas estructura posible en los datos.

### 2.3 Aplicar K-Means 3D y visualizar con Plotly

In [ ]:
# K-Means con 3 variables (usar el K que sugieran codo + silhouette)
K_3d = 6  # ajustar segun los graficos anteriores
km_3d = KMeans(n_clusters=K_3d, random_state=42, n_init=10)
df["cluster_3d"] = km_3d.fit_predict(X_3d)

print(f"Clusters 3D (K={K_3d}):")
print(df["cluster_3d"].value_counts().sort_index())

In [ ]:
# Scatter 3D interactivo con Plotly
fig = px.scatter_3d(
    df, x="income", y="spending", z="Age",
    color=df["cluster_3d"].astype(str),
    title=f"Segmentacion 3D — K-Means (K={K_3d})",
    labels={"income": "Ingreso (k$)", "spending": "Spending Score",
            "Age": "Edad", "color": "Cluster"},
    color_discrete_sequence=["#2563EB", "#C82B40", "#16A34A",
                             "#EA580C", "#7C3AED", "#D97706"],
    opacity=0.7)

fig.update_layout(
    template="plotly_white", width=800, height=600,
    scene=dict(
        xaxis_title="Ingreso (k$)",
        yaxis_title="Spending Score",
        zaxis_title="Edad"))
fig.show()

**Interpretacion:** Roten el grafico con el mouse. La tercera dimension (edad) permite ver separaciones que no eran visibles en 2D. Por ejemplo, dos clientes con el mismo ingreso y gasto pero edades muy diferentes ahora pueden caer en clusters distintos. Plotly permite explorar esto interactivamente.

### 2.4 Profiling 3D — Interpretar los nuevos segmentos

In [ ]:
# Perfil de los clusters 3D
perfil_3d = df.groupby("cluster_3d")[["Age", "income", "spending"]].mean().round(1)
perfil_3d.columns = ["Edad", "Ingreso (k$)", "Spending Score"]
perfil_3d

In [ ]:
# Heatmap de los perfiles (normalizado para comparar)
perfil_3d_norm = (perfil_3d - perfil_3d.min()) / (perfil_3d.max() - perfil_3d.min())

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(perfil_3d_norm, annot=perfil_3d.values, fmt=".1f",
            cmap="RdYlGn", linewidths=1, ax=ax)
ax.set_title("Perfil de cada cluster 3D\n(color = valor normalizado, numero = valor real)")
ax.set_ylabel("Cluster")
plt.tight_layout()
plt.show()

**Interpretacion:** El heatmap muestra de un vistazo que cluster es alto (verde) o bajo (rojo) en cada variable. Los numeros son los valores reales. Este tipo de visualizacion permite identificar rapidamente el "perfil" de cada segmento: por ejemplo, un cluster con edad alta + ingreso alto + gasto bajo es un "jubilado ahorrador".

### 2.5 Comparar 2D vs 3D — Que cambio?

In [ ]:
# Crosstab: como se redistribuyeron los clientes entre 2D y 3D
tabla_comp = pd.crosstab(
    df["segmento"], df["cluster_3d"],
    margins=True, margins_name="Total")
tabla_comp

In [ ]:
# Scatter 2D coloreado por clusters 3D (para ver como se "parten" los grupos 2D)
fig = px.scatter(
    df, x="income", y="spending",
    color=df["cluster_3d"].astype(str),
    symbol="segmento",
    title="Vista 2D coloreada por clusters 3D — Que se partio?",
    labels={"income": "Ingreso (k$)", "spending": "Spending Score", "color": "Cluster 3D"},
    color_discrete_sequence=["#2563EB", "#C82B40", "#16A34A",
                             "#EA580C", "#7C3AED", "#D97706"],
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Al ver los clusters 3D proyectados en 2D, notamos que algunos segmentos originales se "partieron" por la edad. Por ejemplo, el grupo "Mainstream" del centro podria dividirse en jovenes y mayores. Agregar variables enriquece la segmentacion, pero tambien la hace mas compleja de explicar. La decision de cuantas variables usar depende del **caso de negocio**: si la edad importa para las campañas, 3D es mejor; si solo importa el gasto, 2D es suficiente.

### Ejercicio 2 — Nombrar los segmentos 3D (5 min)

Mira el heatmap de la seccion 2.4. Asigna un nombre de negocio a cada cluster 3D usando las 3 variables (edad, ingreso, gasto). Escribe tus nombres en los comentarios.

In [ ]:
# SOLUCION — Ejercicio 2: Nombrar segmentos 3D
perfil_3d_display = df.groupby("cluster_3d")[["Age", "income", "spending"]].mean().round(1)
perfil_3d_display.columns = ["Edad", "Ingreso", "Spending"]

nombres_3d = {}
for c in perfil_3d_display.index:
    edad = perfil_3d_display.loc[c, "Edad"]
    inc  = perfil_3d_display.loc[c, "Ingreso"]
    sp   = perfil_3d_display.loc[c, "Spending"]
    if inc > 70 and sp > 60:
        nombres_3d[c] = f"Premium {'joven' if edad < 35 else 'maduro'}"
    elif inc > 70 and sp < 40:
        nombres_3d[c] = f"Ahorrador {'joven' if edad < 35 else 'senior'}"
    elif inc < 40 and sp > 60:
        nombres_3d[c] = f"Entusiasta {'joven' if edad < 35 else 'maduro'}"
    elif inc < 40 and sp < 40:
        nombres_3d[c] = f"Cauteloso {'joven' if edad < 35 else 'mayor'}"
    else:
        nombres_3d[c] = f"Mainstream {'joven' if edad < 35 else 'maduro'}"

perfil_3d_display["Nombre"] = perfil_3d_display.index.map(nombres_3d)
print(perfil_3d_display[["Nombre", "Edad", "Ingreso", "Spending"]])

---
## 3. DBSCAN — Cuando K-Means falla

K-Means asume que los clusters son **esfericos** (bolitas redondas separadas). Pero los datos reales no siempre tienen esa forma. DBSCAN (Density-Based Spatial Clustering) agrupa puntos por **densidad**: si un punto tiene suficientes vecinos cerca, pertenece a un cluster. Si esta aislado, es **ruido**.

**Dos parametros:**
- `eps`: radio del vecindario. Que tan cerca deben estar dos puntos para ser vecinos.
- `min_samples`: minimo de vecinos para que un punto sea "central" y crezca un cluster.

**Ventajas sobre K-Means:**
- No necesita que elijamos K (lo descubre solo)
- Encuentra clusters de cualquier forma
- Detecta outliers (etiqueta -1)

### 3.1 Cuando K-Means falla: lunas y circulos

In [ ]:
from sklearn.datasets import make_moons, make_circles
from sklearn.cluster import DBSCAN

# Generar datos con forma de lunas
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

# K-Means vs DBSCAN en lunas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Datos reales
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="RdBu", s=20)
axes[0].set_title("Datos reales (2 lunas)", fontweight="bold")

# K-Means
km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons.fit_predict(X_moons),
                cmap="RdBu", s=20)
axes[1].set_title("K-Means (K=2) — FALLA", fontweight="bold", color="#C82B40")

# DBSCAN
db_moons = DBSCAN(eps=0.2, min_samples=5)
labels_db = db_moons.fit_predict(X_moons)
colors_db = ["#C82B40" if l == -1 else ["#2563EB", "#16A34A"][l] for l in labels_db]
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=colors_db, s=20)
axes[2].set_title("DBSCAN — ACIERTA", fontweight="bold", color="#16A34A")

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("K-Means asume clusters esfericos. DBSCAN no.", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

**Interpretacion:** K-Means corta el espacio con lineas rectas — no puede seguir la curvatura de las lunas. DBSCAN sigue la **densidad**: donde hay puntos juntos, hay cluster. No asume ninguna forma.

### 3.2 DBSCAN en Mall Customers — Comparacion con K-Means

In [ ]:
# DBSCAN en los mismos datos de Mall Customers
X_db = StandardScaler().fit_transform(df[["income", "spending"]])

db = DBSCAN(eps=0.45, min_samples=5)
df["dbscan"] = db.fit_predict(X_db)

n_clusters_db = len(set(df["dbscan"]) - {-1})
n_ruido = (df["dbscan"] == -1).sum()
print(f"DBSCAN encontro {n_clusters_db} clusters y {n_ruido} puntos de ruido")
print(f"\nDistribucion:\n{df['dbscan'].value_counts().sort_index()}")

In [ ]:
# Comparacion visual: K-Means vs DBSCAN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
COLORS = ["#2563EB", "#C82B40", "#16A34A", "#EA580C", "#7C3AED", "#D97706"]

# K-Means
ax = axes[0]
for k in range(5):
    mask = df["cluster"] == k
    ax.scatter(df.loc[mask, "income"], df.loc[mask, "spending"],
               c=COLORS[k], s=30, edgecolors="white", linewidths=0.4)
ax.set_xlabel("Ingreso (k$)"); ax.set_ylabel("Spending Score")
ax.set_title("K-Means (K=5)\nTodos asignados", fontweight="bold", color="#2563EB")

# DBSCAN
ax = axes[1]
for k in sorted(df["dbscan"].unique()):
    mask = df["dbscan"] == k
    if k == -1:
        ax.scatter(df.loc[mask, "income"], df.loc[mask, "spending"],
                   c="gray", s=20, marker="x", alpha=0.6, label=f"Ruido ({mask.sum()})")
    else:
        ax.scatter(df.loc[mask, "income"], df.loc[mask, "spending"],
                   c=COLORS[k % len(COLORS)], s=30, edgecolors="white", linewidths=0.4)
ax.set_xlabel("Ingreso (k$)"); ax.set_ylabel("Spending Score")
ax.set_title(f"DBSCAN ({n_clusters_db} clusters + ruido)\nDetecta outliers",
             fontweight="bold", color="#16A34A")
ax.legend()

plt.suptitle("Mismo dataset, dos algoritmos", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

**Interpretacion:** K-Means asigna todos los puntos, incluso los atipicos. DBSCAN detecta los puntos fronterizos como ruido (X grises). Para segmentacion de clientes con grupos compactos, K-Means suele ser mejor. Para deteccion de anomalias o formas irregulares, DBSCAN es superior. **No son competidores, son complementarios.**

| | K-Means | DBSCAN |
|---|---|---|
| Forma de clusters | Esfericos | Cualquier forma |
| Numero de clusters | Tu eliges K | Lo descubre solo |
| Outliers | No los detecta | Los marca como -1 |
| Datos nuevos | `.predict()` funciona | **No tiene** `.predict()` |

---
## 4. Y el cliente nuevo? — De clustering a clasificacion

Entrenamos K-Means, encontramos 5 segmentos, les dimos nombres. El lunes llega un **cliente nuevo** al mall. A que segmento pertenece?

### 4.1 Opcion 1: `km.predict()` — la via rapida

K-Means **si es parametrico**: guarda los centroides. Puede calcular la distancia de un punto nuevo a cada centroide y asignarlo al mas cercano.

In [ ]:
# Cliente nuevo: ingreso 70k, gasto 85
nuevo = [[70, 85]]
segmento_nuevo = km_2d.predict(nuevo)
print(f"Cluster asignado: {segmento_nuevo[0]}")
print(f"Segmento: {nombres[segmento_nuevo[0]]}")

# Visualizar donde cae
fig = px.scatter(
    df, x="income", y="spending", color="segmento",
    title="Cliente nuevo asignado con km.predict()",
    color_discrete_map={
        "Premium": "#16A34A", "Ahorradores VIP": "#2563EB",
        "Entusiastas": "#EA580C", "Cautelosos": "#7C3AED",
        "Mainstream": "#6B7280"},
    opacity=0.5)
fig.add_trace(go.Scatter(
    x=[70], y=[85], mode="markers",
    marker=dict(size=20, color="red", symbol="star", line=dict(width=2, color="black")),
    name="CLIENTE NUEVO"))
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Funciona directo con K-Means. Pero DBSCAN **no tiene `.predict()`**. Y a veces queremos usar variables extras (genero, ciudad) que no estaban en el clustering. Necesitamos una solucion mas general.

### 4.2 Opcion 2: Clasificador encima — la via general

La idea: usar los clusters descubiertos como **etiqueta (y)** y entrenar un modelo supervisado. Ahora si podemos predecir clientes nuevos con cualquier algoritmo de clustering.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 1. Los clusters descubiertos son ahora nuestro "y"
y_segmentos = df["cluster"]

# 2. Entrenar un clasificador con TODAS las variables (incluso las que no usamos en clustering)
X_clf = df[["Age", "income", "spending"]]
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_clf, y_segmentos)

# 3. Que tan bien clasifica? (cross-validation)
scores = cross_val_score(clf, X_clf, y_segmentos, cv=5, scoring="accuracy")
print(f"Accuracy CV: {scores.mean():.3f} +/- {scores.std():.3f}")

# 4. Predecir cliente nuevo (ahora podemos usar Age tambien!)
nuevo_cliente = pd.DataFrame([{"Age": 28, "income": 70, "spending": 85}])
pred = clf.predict(nuevo_cliente)
print(f"\nCliente nuevo (28 anos, ingreso 70k, gasto 85):")
print(f"  Segmento predicho: {nombres[pred[0]]}")

**Interpretacion:** Esto conecta todo el diplomado:
1. **No supervisado** descubre los grupos (K-Means/DBSCAN)
2. Los clusters se convierten en **etiquetas** (y)
3. **Supervisado** entrena un clasificador encima
4. Ahora podemos **predecir** clientes nuevos

Ventajas de esta via: funciona con cualquier algoritmo de clustering, puede usar variables extras, y se puede meter dentro de un Pipeline.

---
## 5. Ejercicio: Descubrir especies en datos sin etiquetas

Un biologo te entrega mediciones de 150 flores. Tiene 4 medidas por flor (largo y ancho de sepalo y petalo) pero **no sabe cuantas especies hay**. Tu mision: usar K-Means para descubrir los grupos ocultos.

### 5.1 Cargar datos (sin etiquetas)

In [ ]:
# Cargar Iris SIN mirar las etiquetas
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df_flores = iris.data.copy()  # SOLO las medidas, sin la especie

print(f"Shape: {df_flores.shape}")
print(f"Variables: {list(df_flores.columns)}")
df_flores.head()

### 5.2 Tu turno — Descubrir cuantas especies hay (15 min)

Instrucciones:
1. Escala los datos con `StandardScaler` (las 4 variables tienen escalas distintas).
2. Usa el **metodo del codo** y **Silhouette Score** para encontrar el K optimo.
3. Aplica K-Means con ese K.
4. Crea un scatter con `petal length (cm)` vs `petal width (cm)` coloreado por cluster.
5. **Pregunta clave:** Cuantas especies descubrio K-Means?

In [ ]:
# SOLUCION — Ejercicio Iris
scaler_iris = StandardScaler()
X_iris = scaler_iris.fit_transform(df_flores)

inertias_iris = []
sil_iris = []
K_iris = range(2, 8)
for k in K_iris:
    km_i = KMeans(n_clusters=k, random_state=42, n_init=10)
    labs = km_i.fit_predict(X_iris)
    inertias_iris.append(km_i.inertia_)
    sil_iris.append(silhouette_score(X_iris, labs))

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Metodo del Codo — Iris", "Silhouette Score — Iris"))
fig.add_trace(go.Scatter(x=list(K_iris), y=inertias_iris, mode="lines+markers",
    marker=dict(color="#2563EB")), row=1, col=1)
fig.add_trace(go.Scatter(x=list(K_iris), y=sil_iris, mode="lines+markers",
    marker=dict(color="#C82B40")), row=1, col=2)
fig.update_xaxes(title_text="K", row=1, col=1)
fig.update_xaxes(title_text="K", row=1, col=2)
fig.update_yaxes(title_text="Inercia", row=1, col=1)
fig.update_yaxes(title_text="Silhouette", row=1, col=2)
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

km_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
df_flores["cluster"] = km_iris.fit_predict(X_iris)

fig = px.scatter(df_flores, x="petal length (cm)", y="petal width (cm)",
    color=df_flores["cluster"].astype(str),
    title="Clusters descubiertos por K-Means (sin ver etiquetas)",
    color_discrete_sequence=["#2563EB", "#C82B40", "#16A34A"], opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()
print(f"\nK-Means descubrio {df_flores['cluster'].nunique()} grupos")

### 5.3 Verificacion — Comparar con las especies reales

Ahora **si** miramos las etiquetas para verificar si K-Means descubrio las especies.

In [ ]:
# SOLUCION — Verificacion
df_flores["especie_real"] = iris.target.map({0: "setosa", 1: "versicolor", 2: "virginica"})
tabla = pd.crosstab(df_flores["cluster"], df_flores["especie_real"])
print("Clusters vs Especies reales:")
print(tabla)

fig = px.scatter(df_flores, x="petal length (cm)", y="petal width (cm)",
    color="especie_real", title="Especies REALES (para comparar con K-Means)",
    color_discrete_map={"setosa": "#2563EB", "versicolor": "#C82B40", "virginica": "#16A34A"},
    opacity=0.8)
fig.update_layout(template="plotly_white")
fig.show()
print("\nSetosa: K-Means la separa perfectamente")
print("Versicolor y Virginica: hay algo de confusion")

**Interpretacion:** K-Means deberia encontrar 2 o 3 clusters. Setosa siempre se separa perfectamente. Versicolor y Virginica se confunden un poco — esas dos especies son parecidas en sus medidas. Esto demuestra que K-Means puede descubrir estructura real en datos sin etiquetas, pero no es perfecto cuando los grupos se solapan.

---
## 6. Pipelines de sklearn — Encadenar pasos sin errores

Hasta ahora hemos hecho los pasos de ML por separado:

```python
# Paso 1: escalar
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Paso 2: entrenar modelo
modelo = RandomForestClassifier()
modelo.fit(X_train_s, y_train)

# Paso 3: predecir
y_pred = modelo.predict(X_test_s)
```

Esto funciona, pero tiene **dos problemas**:
1. **Facil olvidar un paso** — si olvidas escalar X_test, el modelo recibe datos en otra escala y predice basura.
2. **Data leakage** — si haces `scaler.fit_transform(X)` antes del split, el scaler "ve" datos de test durante el entrenamiento.

**Pipeline resuelve ambos problemas**: encadena los pasos en un solo objeto.

### 6.1 Tu primer Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.datasets import fetch_openml

# Cargar diabetes (mismo dataset de clase 18)
d = fetch_openml("diabetes", version=1, as_frame=True, parser="auto")
df_diab = d.frame.copy()
df_diab["target"] = (df_diab["class"] == "tested_positive").astype(int)
df_diab = df_diab.drop(columns=["class"])

for col in ["plas", "pres", "skin", "insu", "mass"]:
    df_diab[col] = df_diab[col].replace(0, np.nan).fillna(df_diab[col].median())

X = df_diab.drop(columns=["target"])
y = df_diab["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# SIN Pipeline (como lo haciamos antes)
scaler_old = StandardScaler()
X_train_s = scaler_old.fit_transform(X_train)
X_test_s = scaler_old.transform(X_test)

lr_old = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_old.fit(X_train_s, y_train)

y_prob_old = lr_old.predict_proba(X_test_s)[:, 1]
print(f"AUC (sin pipeline): {roc_auc_score(y_test, y_prob_old):.3f}")

In [ ]:
# CON Pipeline — mismo resultado, menos codigo, sin errores
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

# Un solo .fit() escala Y entrena
pipe_lr.fit(X_train, y_train)

# Un solo .predict_proba() escala Y predice
y_prob_pipe = pipe_lr.predict_proba(X_test)[:, 1]
print(f"AUC (con pipeline): {roc_auc_score(y_test, y_prob_pipe):.3f}")

**Interpretacion:** Mismo AUC, pero el codigo es mas limpio y seguro. El Pipeline garantiza que:
- `fit()` aplica `fit_transform` al scaler y `fit` al modelo (en ese orden).
- `predict()` aplica `transform` al scaler y `predict` al modelo.
- **Nunca** puedes olvidar escalar los datos de test.

### 6.2 Pipeline + Cross-Validation (sin data leakage)

El gran beneficio: cuando usamos Pipeline dentro de `cross_val_score`, el scaler se ajusta **solo en los folds de entrenamiento** de cada iteracion. Esto previene data leakage.

In [ ]:
# Cross-validation con Pipeline — el scaler NO ve los datos de validacion
scores_pipe = cross_val_score(pipe_lr, X_train, y_train, cv=5, scoring="roc_auc")

print(f"AUC por fold: {scores_pipe.round(3)}")
print(f"AUC promedio: {scores_pipe.mean():.3f} +/- {scores_pipe.std():.3f}")

**Interpretacion:** En cada fold, el Pipeline:
1. Ajusta el scaler en los 4 folds de train → `fit_transform`
2. Transforma el fold de validacion → `transform` (sin fit!)
3. Entrena el modelo en los datos escalados de train
4. Evalua en los datos escalados de validacion

Esto es exactamente lo correcto. Sin Pipeline, es facil escalar todo antes del CV y contaminar la evaluacion.

### 6.3 Pipeline + GridSearchCV

Podemos buscar hiperparametros **dentro** del Pipeline. La sintaxis usa doble guion bajo: `nombre_paso__parametro`.

In [ ]:
# Pipeline con Random Forest + GridSearchCV
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", RandomForestClassifier(class_weight="balanced", random_state=42))
])

# Los parametros del grid usan la notacion: nombre_paso__parametro
param_grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [5, 8, None],
}

grid = GridSearchCV(pipe_rf, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Mejor AUC (CV):     {grid.best_score_:.3f}")
print(f"Mejores parametros: {grid.best_params_}")

In [ ]:
# Evaluar en test con el mejor pipeline
mejor_pipe = grid.best_estimator_
y_prob_best = mejor_pipe.predict_proba(X_test)[:, 1]
y_pred_best = mejor_pipe.predict(X_test)

print(f"AUC test: {roc_auc_score(y_test, y_prob_best):.3f}")
print()
print(classification_report(y_test, y_pred_best,
                            target_names=["No Diabetes", "Diabetes"]))

**Interpretacion:** `grid.best_estimator_` devuelve el Pipeline completo ya entrenado con los mejores hiperparametros. Cuando llamamos `predict_proba(X_test)`, el Pipeline **automaticamente escala** los datos de test antes de pasarlos al modelo. Zero posibilidad de error.

### 6.4 Visualizar el Pipeline

In [ ]:
# sklearn puede dibujar el pipeline como diagrama
from sklearn import set_config
set_config(display="diagram")

mejor_pipe

**Interpretacion:** sklearn puede renderizar el Pipeline como un diagrama visual. Esto es util para documentar y comunicar exactamente que pasos sigue el modelo. Hagan clic en cada paso para ver sus hiperparametros.

### 6.5 Comparar Logistica vs RF con Pipelines

In [ ]:
# Comparar multiples modelos con pipelines y CV
pipelines = {
    "Logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", RandomForestClassifier(n_estimators=200, max_depth=8,
                                          class_weight="balanced", random_state=42))
    ]),
}

resultados = []
for nombre, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    resultados.append({
        "Modelo": nombre,
        "AUC mean": f"{scores.mean():.3f}",
        "AUC std": f"{scores.std():.3f}",
        "Folds": str(scores.round(3))
    })
    print(f"{nombre}: {scores.mean():.3f} +/- {scores.std():.3f}")

pd.DataFrame(resultados)

**Interpretacion:** Con Pipelines, comparar modelos se reduce a un loop limpio. Cada modelo tiene su propio flujo de preprocesamiento encapsulado. No hay riesgo de mezclar transformaciones entre modelos.

### 6.6 Pipeline para clustering (K-Means)

In [ ]:
# Pipeline para clustering: Scaler + KMeans
pipe_km = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=5, random_state=42, n_init=10))
])

# fit_predict en un solo paso — el scaler se aplica automaticamente
clusters = pipe_km.fit_predict(df[["income", "spending", "Age"]])
print(f"Clusters asignados: {np.unique(clusters)}")
print(f"Silhouette Score: {silhouette_score(pipe_km['scaler'].transform(df[['income', 'spending', 'Age']]), clusters):.3f}")

**Interpretacion:** Pipelines no son solo para clasificacion — tambien funcionan con clustering. `pipe_km.fit_predict(datos)` escala y agrupa en una sola linea. Si mañana cambiamos `StandardScaler` por `MinMaxScaler`, solo modificamos una linea y todo sigue funcionando.

### Ejercicio 3 — Pipeline con DecisionTree (10 min)

1. Crea un Pipeline con `StandardScaler` + `DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)`.
2. Evalua con `cross_val_score` (cv=5, scoring="roc_auc") en el dataset de diabetes.
3. Compara su AUC con la Logistica y el Random Forest de la seccion 4.5.
4. Cual modelo gana? Tiene sentido que el arbol tenga o no el scaler?

In [ ]:
# SOLUCION — Ejercicio 3: Pipeline con DecisionTree
from sklearn.tree import DecisionTreeClassifier

pipe_dt = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42))
])

scores_dt = cross_val_score(pipe_dt, X_train, y_train, cv=5, scoring="roc_auc")
print(f"Decision Tree: {scores_dt.mean():.3f} +/- {scores_dt.std():.3f}")

print("\n=== Comparacion ===")
for nombre, pipe in pipelines.items():
    s = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{nombre}: {s.mean():.3f} +/- {s.std():.3f}")
print(f"Decision Tree: {scores_dt.mean():.3f} +/- {scores_dt.std():.3f}")

print("\nTiene sentido el scaler con arbol?")
print("NO. Los arboles usan splits, no distancias. El scaler no perjudica pero no es necesario.")

---
## Resumen de la clase

| Tema | Concepto clave |
|---|---|
| **Profiling de clusters** | Calcular promedios por cluster y asignar nombres de negocio |
| **Radar chart** | Visualizacion multivariable que muestra la "forma" de cada segmento |
| **Recomendaciones** | La segmentacion debe generar acciones concretas, no solo numeros |
| **K-Means 3D** | Agregar variables enriquece la segmentacion pero puede cambiar el K optimo |
| **StandardScaler** | Indispensable cuando las variables tienen escalas distintas |
| **DBSCAN** | Clusters de cualquier forma, no necesita K, detecta outliers |
| **K-Means vs DBSCAN** | Complementarios: K-Means para bolitas, DBSCAN para formas irregulares |
| **Cliente nuevo** | `km.predict()` o entrenar un clasificador encima de los clusters |
| **Pipeline** | Encadena preprocesamiento + modelo en un solo objeto |
| **Pipeline + CV** | Previene data leakage: el scaler se ajusta solo en los folds de train |
| **Pipeline + GridSearchCV** | Busca hiperparametros con notacion `paso__parametro` |